In [9]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from vmdpy import VMD

In [10]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [11]:
from configs import config

# Project Root
PROJECT_ROOT = config.PROJECT_ROOT

# Data Paths
RAW_DATA_PATH = config.RAW_DATA_FILE
PROCESSED_DATA_PATH = config.PROCESSED_DATA_FILE

# Artifact Directory
ARTIFACT_DIR = config.ARTIFACT_DIR
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(f"Missing processed data file: {PROCESSED_DATA_PATH}")

In [12]:
df = pd.read_csv(
    PROCESSED_DATA_PATH,            
    parse_dates=["timestamp"]
)

print(f"Dataset Shape : {df.shape}")

df.head()

Dataset Shape : (1065289, 8)


,timestamp,Active_Power,Weather_Temperature_Celsius,Weather_Relative_Humidity,Global_Horizontal_Radiation,Diffuse_Horizontal_Radiation,Radiation_Global_Tilted,Radiation_Diffuse_Tilted
0,2014-01-01 00:00:00,-0.725207,30.081875,12.679476,1.267926,0.587425,0.928665,1.656146
1,2014-01-01 00:05:00,-0.717201,29.945786,12.650104,2.656051,1.286431,1.383545,1.948417
2,2014-01-01 00:10:00,-0.694308,29.381653,12.909550,4.119638,2.647693,2.077302,2.563462
3,2014-01-01 00:15:00,-0.682230,28.800579,13.392023,4.079689,2.499754,2.314080,2.791505
4,2014-01-01 00:20:00,-0.676284,28.300385,13.731309,4.032659,2.465453,2.441365,2.923790


In [13]:
TARGET_COLUMN = "Active_Power"

FEATURE_COLUMNS = [
    "Weather_Temperature_Celsius",
    "Weather_Relative_Humidity",
    "Global_Horizontal_Radiation",
    "Diffuse_Horizontal_Radiation",
    "Radiation_Global_Tilted",
    "Radiation_Diffuse_Tilted",
    "Active_Power",          
]

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

print("Feature Shape :", X.shape)
print("Target Shape  :", y.shape)

Feature Shape : (1065289, 7)
Target Shape  : (1065289,)


In [14]:
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

n_samples = len(df)

train_end = int(TRAIN_RATIO * n_samples)
val_end = train_end + int(VAL_RATIO * n_samples)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

print(f"Train Samples      : {len(X_train)}")
print(f"Validation Samples : {len(X_val)}")
print(f"Test Samples       : {len(X_test)}")

Train Samples      : 745702
Validation Samples : 159793
Test Samples       : 159794


In [15]:
# --- VMD Decomposition of Historical PV Power (per split, no leakage) ---
# Each split is decomposed independently, following AI_VMD-HS_CNN-BiLSTM-A (Sec. 3.3):
# "the partitioned training and test sets are decomposed ... respectively,
#  which avoids problems such as data leakage due to improper decomposition."

VMD_ALPHA = 2000   # bandwidth constraint — standard default in VMD-PV literature
VMD_TAU = 0.        # noise tolerance — 0 enforces exact signal reconstruction
VMD_K = 5           # number of modes — matches the AIC-optimal K found for this
                     # same Alice Springs PV dataset in the reference paper (Table 3)
VMD_DC = 0           # no imposed zero-frequency (DC) mode
VMD_INIT = 1         # uniform initialization of center frequencies
VMD_TOL = 1e-7


def decompose_active_power(power_series, K=VMD_K):
    """
    Apply VMD to a single, already-split Active_Power series.
    Returns a DataFrame of K IMF columns aligned to the input index.
    """
    power_values = power_series.to_numpy()

    if np.isnan(power_values).any():
        # VMD cannot handle missing values: it operates on the whole signal's
        # spectrum at once, so a single NaN corrupts the entire decomposition.
        # Interpolation is restricted to this split only (no cross-split fill).
        power_values = (
            pd.Series(power_values)
            .interpolate(method="linear", limit_direction="both")
            .to_numpy()
        )

    imfs, _, _ = VMD(power_values, VMD_ALPHA, VMD_TAU, K, VMD_DC, VMD_INIT, VMD_TOL)

    imf_columns = [f"Active_Power_IMF{i + 1}" for i in range(K)]
    imf_df = pd.DataFrame(imfs.T, columns=imf_columns, index=power_series.index)

    return imf_df


train_imfs = decompose_active_power(X_train["Active_Power"])
val_imfs = decompose_active_power(X_val["Active_Power"])
test_imfs = decompose_active_power(X_test["Active_Power"])

IMF_COLUMNS = list(train_imfs.columns)

X_train = pd.concat([X_train.drop(columns=["Active_Power"]), train_imfs], axis=1)
X_val = pd.concat([X_val.drop(columns=["Active_Power"]), val_imfs], axis=1)
X_test = pd.concat([X_test.drop(columns=["Active_Power"]), test_imfs], axis=1)

FEATURE_COLUMNS = [c for c in FEATURE_COLUMNS if c != "Active_Power"] + IMF_COLUMNS

print(f"IMF columns added   : {IMF_COLUMNS}")
print(f"Updated feature list: {FEATURE_COLUMNS}")

MemoryError: Unable to allocate 55.6 GiB for an array with shape (500, 1491404, 5) and data type complex128

In [ ]:
feature_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)

X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

In [ ]:
target_scaler = StandardScaler()

y_train_scaled = target_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1))

y_val_scaled = target_scaler.transform(y_val.to_numpy().reshape(-1, 1))
y_test_scaled = target_scaler.transform(y_test.to_numpy().reshape(-1, 1))

In [ ]:
LOOKBACK = 24          # Previous 2 hours
HORIZONS = [3, 12]     # 15 min and 60 min
STRIDE = 1

In [ ]:
def create_sequences(features, target, lookback, horizon, stride=1):
    
    X_seq = []
    y_seq = []

    last_start = len(features) - lookback - horizon + 1

    for start in range(0, last_start, stride):

        end = start + lookback

        X_seq.append(features[start:end])

        y_seq.append(target[end:end + horizon, 0])

    X_seq = torch.tensor(np.array(X_seq), dtype=torch.float32)
    y_seq = torch.tensor(np.array(y_seq), dtype=torch.float32)

    return X_seq, y_seq

In [ ]:
datasets = {}

for horizon in HORIZONS:

    X_train_seq, y_train_seq = create_sequences(
        X_train_scaled,
        y_train_scaled,
        LOOKBACK,
        horizon,
        STRIDE,
    )

    X_val_seq, y_val_seq = create_sequences(
        X_val_scaled,
        y_val_scaled,
        LOOKBACK,
        horizon,
        STRIDE,
    )

    X_test_seq, y_test_seq = create_sequences(
        X_test_scaled,
        y_test_scaled,
        LOOKBACK,
        horizon,
        STRIDE,
    )

    datasets[horizon] = {
        "train": (X_train_seq, y_train_seq),
        "val": (X_val_seq, y_val_seq),
        "test": (X_test_seq, y_test_seq),
    }

    print(f"\n===== {horizon * 5} Minute Forecast =====")
    print("Train :", X_train_seq.shape, y_train_seq.shape)
    print("Val   :", X_val_seq.shape, y_val_seq.shape)
    print("Test  :", X_test_seq.shape, y_test_seq.shape)


===== 15 Minute Forecast =====
Train : torch.Size([745676, 24, 6]) torch.Size([745676, 3])
Val   : torch.Size([159767, 24, 6]) torch.Size([159767, 3])
Test  : torch.Size([159768, 24, 6]) torch.Size([159768, 3])

===== 60 Minute Forecast =====
Train : torch.Size([745667, 24, 6]) torch.Size([745667, 12])
Val   : torch.Size([159758, 24, 6]) torch.Size([159758, 12])
Test  : torch.Size([159759, 24, 6]) torch.Size([159759, 12])


In [ ]:
for horizon in HORIZONS:

    torch.save(
        {
            "X": datasets[horizon]["train"][0],
            "y": datasets[horizon]["train"][1],
        },
        ARTIFACT_DIR / f"train_{horizon * 5}.pt",
    )

    torch.save(
        {
            "X": datasets[horizon]["val"][0],
            "y": datasets[horizon]["val"][1],
        },
        ARTIFACT_DIR / f"val_{horizon * 5}.pt",
    )

    torch.save(
        {
            "X": datasets[horizon]["test"][0],
            "y": datasets[horizon]["test"][1],
        },
        ARTIFACT_DIR / f"test_{horizon * 5}.pt",
    )

In [ ]:
with open(ARTIFACT_DIR / "feature_scaler.pkl", "wb") as f:
    pickle.dump(feature_scaler, f)

with open(ARTIFACT_DIR / "target_scaler.pkl", "wb") as f:
    pickle.dump(target_scaler, f)

In [ ]:
config = {
    "lookback": LOOKBACK,
    "horizons": HORIZONS,
    "stride": STRIDE,
    "features": FEATURE_COLUMNS,
    "target": TARGET_COLUMN,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "feature_scaler": "StandardScaler",
    "target_scaler": "StandardScaler",
    "vmd_alpha": VMD_ALPHA,     
    "vmd_tau": VMD_TAU,         
    "vmd_K": VMD_K,             
    "vmd_DC": VMD_DC,           
}

with open(ARTIFACT_DIR / "preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=4)

In [ ]:
# --- Sanity check: verify no timestamp overlap between feature window and target ---

def verify_no_leakage(timestamps, lookback, horizon, stride=1, n_samples_to_check=5):
    """
    Mirrors create_sequences' indexing logic but on timestamps,
    so we can literally see which dates/times land in features vs target.
    """
    last_start = len(timestamps) - lookback - horizon + 1

    checked = 0
    for start in range(0, last_start, stride):
        end = start + lookback

        feature_timestamps = timestamps[start:end]
        target_timestamps = timestamps[end:end + horizon]

        last_feature_time = feature_timestamps[-1]
        first_target_time = target_timestamps[0]

        assert last_feature_time < first_target_time, (
            f"LEAKAGE DETECTED at start={start}: "
            f"last feature time {last_feature_time} is not before "
            f"first target time {first_target_time}"
        )

        if checked < n_samples_to_check:
            print(f"Sample start={start}:")
            print(f"  Feature window : {feature_timestamps[0]}  ...  {last_feature_time}")
            print(f"  Target window  : {first_target_time}  ...  {target_timestamps[-1]}")
            print(f"  Gap OK: last feature < first target -> {last_feature_time < first_target_time}\n")
            checked += 1

    print(f"Checked all {last_start} windows for horizon={horizon}. No overlap found.")


# Run it on the train split's actual timestamps, for each horizon
train_timestamps = df["timestamp"].iloc[:train_end].to_numpy()

for horizon in HORIZONS:
    print(f"\n===== Verifying horizon={horizon} ({horizon * 5} min) =====")
    verify_no_leakage(train_timestamps, LOOKBACK, horizon, STRIDE)

In [ ]:
print("Artifacts Saved Successfully\n")

for file in sorted(ARTIFACT_DIR.iterdir()):
    print(file.name)

Artifacts Saved Successfully

feature_scaler.pkl
preprocessing_config.json
target_scaler.pkl
test_15.pt
test_60.pt
train_15.pt
train_60.pt
val_15.pt
val_60.pt
